# Generate API Payloads from Held-Out Records

Converts **real LendingClub records from the held-out test split** into the JSON payloads
the FastAPI endpoints expect, scores them with the deployed model artefacts, and exercises
all three serving arms end to end.

This is the inference-layer demonstration: it shows that a genuine record flows correctly
through the fitted preprocessor, the model, and each of the three architectures.

`data/X_test.parquet` is committed, so this notebook runs from a fresh clone. The synthetic
500-applicant benchmark pool is a separate thing — see `tests/payloads.py`.

In [12]:
import json
import time
from pathlib import Path
from urllib.request import Request, urlopen

import joblib
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "artifacts").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

X_test = pd.read_parquet(ROOT / "data" / "X_test.parquet")
preprocessor = joblib.load(ROOT / "artifacts" / "realtime_preprocessor.joblib")
model = joblib.load(ROOT / "artifacts" / "lightgbm_credit_model.joblib")

API_BASE = "http://localhost:8000"
print(f"held-out records : {len(X_test):,} x {len(X_test.columns)} features")

held-out records : 269,070 x 86 features


## Record to payload

The API accepts raw applicant fields as JSON and applies the fitted `ColumnTransformer`
itself, so a payload is just one record with NumPy types converted and NaNs turned into
`null`.

In [17]:
def to_payload(row: pd.Series) -> dict:
    return {
        col: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v))
        for col, v in row.items()
    }


payload = to_payload(X_test.iloc[0])
print(f"{len(payload)} fields\n")
print(json.dumps(dict(list(payload.items())), indent=2))

86 fields

{
  "loan_amnt": 11925.0,
  "term": 36.0,
  "int_rate": 6.239999771118164,
  "installment": 364.0799865722656,
  "grade": "A",
  "sub_grade": "A2",
  "emp_length": "2 years",
  "home_ownership": "MORTGAGE",
  "annual_inc": 47819.0,
  "verification_status": "Not Verified",
  "purpose": "credit_card",
  "addr_state": "MD",
  "dti": 22.190000534057617,
  "delinq_2yrs": 0.0,
  "fico_range_low": 760.0,
  "fico_range_high": 764.0,
  "inq_last_6mths": 0.0,
  "mths_since_last_delinq": 999.0,
  "mths_since_last_record": 999.0,
  "open_acc": 12.0,
  "pub_rec": 0.0,
  "revol_bal": 11935.0,
  "revol_util": 35.79999923706055,
  "total_acc": 41.0,
  "collections_12_mths_ex_med": 0.0,
  "mths_since_last_major_derog": 999.0,
  "policy_code": 1.0,
  "application_type": "Individual",
  "acc_now_delinq": 0.0,
  "tot_coll_amt": 0.0,
  "tot_cur_bal": 250376.0,
  "open_acc_6m": null,
  "open_act_il": null,
  "open_il_12m": null,
  "open_il_24m": null,
  "mths_since_rcnt_il": null,
  "total_bal_il

## Risk distribution of real applicants

Scoring a sample locally with the same artefacts the API loads. Tier-2 is gated to
Medium/High risk (default probability >= 0.40), so this is the fraction of real traffic
that would trigger background explanation work.

In [14]:
sample = X_test.head(1000)
probs = model.predict_proba(preprocessor.transform(sample))[:, 1]
tiers = pd.cut(probs, [-0.01, 0.40, 0.60, 1.01], labels=["Low", "Medium", "High"])

for tier, count in tiers.value_counts().reindex(["Low", "Medium", "High"]).items():
    print(f"  {tier:<7}{count:>5}   {count / len(sample):>6.1%}")
print(f"\nwould trigger Tier-2: {(probs >= 0.40).mean():.1%}")

  Low      428    42.8%
  Medium   321    32.1%
  High     251    25.1%

would trigger Tier-2: 57.2%


## Smoke test — all three arms

Requires the stack running (`docker compose up -d`). Set `RUN_REQUEST = False` to skip.

A Medium/High-risk applicant is used so the Tier-2 path is actually exercised: the
synchronous arm computes it inline, the asynchronous arm defers it to the workers.

In [15]:
RUN_REQUEST = True


def post(path: str, body: dict, timeout: int = 60) -> dict:
    request = Request(
        f"{API_BASE}{path}",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)


task_id = None
if RUN_REQUEST:
    idx = next(i for i, t in enumerate(tiers) if t in ("Medium", "High"))
    applicant = to_payload(sample.iloc[idx])

    for arm in ("baseline", "synch", "asynch"):
        result = post(f"/predict/{arm}", applicant)
        print(f"  {arm:<9}{result['api_latency_ms']:>9.1f} ms   tier={result['risk_tier']}")
        if arm == "asynch":
            task_id = result["deep_analysis"]["task_id"]
else:
    print("Set RUN_REQUEST = True with the stack running.")

  baseline      86.2 ms   tier=Medium
  synch        683.9 ms   tier=Medium
  asynch       128.9 ms   tier=Medium


## Collecting the deferred Tier-2 explanation

The asynchronous arm returned a `task_id` instead of the deep analysis. Polling
`/result/{task_id}` until the background workers finish it.

In [16]:
if task_id:
    started = time.time()
    for _ in range(60):
        status = json.load(urlopen(f"{API_BASE}/result/{task_id}", timeout=10))
        if status["status"] == "Completed":
            print(f"Tier-2 ready after {time.time() - started:.1f}s")
            print("keys:", list(status["data"].keys()))
            break
        time.sleep(1)
    else:
        print("Tier-2 did not complete within 60s")

Tier-2 ready after 1.0s
keys: ['complex_interactions', 'actionable_recourse', 'background_processing_time_ms']
